# CPG-RL 論文標準版訓練 Notebook（Go2 · MJX · Colab GPU）

比 5 維起步版更貼近 Bellegarda & Ijspeert (2022) 原論文：
- **動作 12 維**：每條腿 (μx, μy, ω)。μ∈[1,2]、ω∈[0,4.5]Hz。
- **2D 腳掌**：含側向 y（用髖外展），不只矢狀面。
- **腿間耦合振盪器**：相位用 Kuramoto 耦合鎖成 trot（可放寬成學步態）。
- **固定離地 g_c**：抬腳高度是常數、不乘振幅 → **結構上解決拖地**（策略沒得偷懶）。
- **觀測含腳觸地布林 + 每腿 CPG 狀態**。

其餘（PD 對齊 kp=90/kd=3、domain randomization、Brax PPO、存權重）與 5 維版相同。
本檔的 CPG 數學已先在本機開迴路驗證：前進、腳抬起 ≈5cm、不跌倒。

> ⚠️ 未在 GPU 實跑，MJX/brax 版本可能有 API 差異；務必先跑 Smoke test。
> 用 `scene_mjx.xml`（位置伺服），我們用 `apply_pd()` 改成等效 kp=90/kd=3。

## 第 1 步：GPU + 安裝

執行階段 → 變更類型 → GPU。`MUJOCO_GL=egl` 在 import mujoco 前先設，之後影片格才渲染得動。

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
!pip install -q mujoco mujoco-mjx brax mediapy
print("done")

In [ ]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())   # 要看到 cuda

## 第 1.5 步：Go2 模型

In [ ]:
import os, subprocess
if not os.path.exists("mujoco_menagerie"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/google-deepmind/mujoco_menagerie.git"], check=True)
SCENE = "mujoco_menagerie/unitree_go2/scene_mjx.xml"
print("model exists:", os.path.exists(SCENE))

## 第 2 步：論文版 CPG（JAX）

每腿狀態：振幅 rx, ry（二階動力學收斂到 μx, μy）與相位 θ。相位用**腿間耦合**鎖 trot。
腳掌偏移（相對 home 腳位）：
- `dx = -d·f(rx)·cos θ`（前後）、`dy = d·f(ry)·cos θ`（側向），`f(r)=2(r-1)-1 ∈[-1,1]`
- `dz = g_c·sin θ`(擺動, sin>0) / `g_p·sin θ`(站立) ← **g_c 固定、不乘振幅**

In [ ]:
import jax.numpy as jnp

MU_MIN, MU_MAX = 1.0, 2.0
OMEGA_MIN, OMEGA_MAX = 0.0, 4.5
A_CONV = 50.0
D_STEP = 0.12          # 步幅尺度
G_C = 0.08             # 擺動離地（固定）★
G_P = 0.01             # 站立下壓
W_COUP = 8.0           # 腿間耦合強度
N_CPG_SUB = 4
PHASE_OFFSET = jnp.array([0.0, jnp.pi, jnp.pi, 0.0])          # trot FL,FR,RL,RR
PHI = PHASE_OFFSET[None, :] - PHASE_OFFSET[:, None]           # (4,4) 目標相位差


def cpg_init():
    return {"rx": jnp.full(4, 1.5), "rx_d": jnp.zeros(4),
            "ry": jnp.full(4, 1.5), "ry_d": jnp.zeros(4),
            "theta": PHASE_OFFSET}


def cpg_step(c, mux, muy, omega, dt):
    rx, rxd, ry, ryd, th = c["rx"], c["rx_d"], c["ry"], c["ry_d"], c["theta"]
    h = dt / N_CPG_SUB
    for _ in range(N_CPG_SUB):
        rxd = rxd + A_CONV * (A_CONV / 4.0 * (mux - rx) - rxd) * h
        rx = rx + rxd * h
        ryd = ryd + A_CONV * (A_CONV / 4.0 * (muy - ry) - ryd) * h
        ry = ry + ryd * h
        rbar = 0.5 * (rx + ry)
        diff = th[None, :] - th[:, None] - PHI              # (4,4)
        coup = jnp.sum(rbar[None, :] * jnp.sin(diff), axis=1)
        th = th + (2.0 * jnp.pi * omega + W_COUP * coup) * h
    th = jnp.mod(th, 2.0 * jnp.pi)
    return {"rx": rx, "rx_d": rxd, "ry": ry, "ry_d": ryd, "theta": th}


def action_to_cpg_cmd(action):
    a = jnp.tanh(action).reshape(4, 3)
    mux = (a[:, 0] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    muy = (a[:, 1] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    omega = (a[:, 2] + 1) / 2 * (OMEGA_MAX - OMEGA_MIN) + OMEGA_MIN
    return mux, muy, omega


def cpg_foot_offsets(c):
    th = c["theta"]
    fx = 2 * (c["rx"] - MU_MIN) / (MU_MAX - MU_MIN) - 1.0
    fy = 2 * (c["ry"] - MU_MIN) / (MU_MAX - MU_MIN) - 1.0
    dx = -D_STEP * fx * jnp.cos(th)
    dy = D_STEP * fy * jnp.cos(th)
    dz = jnp.where(jnp.sin(th) > 0, G_C * jnp.sin(th), G_P * jnp.sin(th))
    return jnp.stack([dx, dy, dz], axis=-1)                 # (4,3)


def cpg_to_joint_targets(c, f0s, jinvs, home3):
    off = cpg_foot_offsets(c)                               # (4,3)
    dq = jnp.einsum("kij,kj->ki", jinvs, off)               # (4,3)
    q = home3[None, :] + dq
    return q.reshape(12)

_c = cpg_init()
_c = cpg_step(_c, jnp.full(4, 1.8), jnp.full(4, 1.5), jnp.full(4, 2.0), 0.02)
print("cpg ok, theta=", _c["theta"])

## 第 3 步：每腿 3D IK 常數

對每條腿，在 home 姿態用有限差分求 3x3 Jacobian：腳掌(x,y,z 相對 hip) 對 (hip外展, thigh, calf)。
左右腿的外展方向不同，逐腿各算一份 Jinv 就自動處理好正負。

In [ ]:
import numpy as np, mujoco

LEGS = ["FL", "FR", "RL", "RR"]
HOME3_np = np.array([0.0, 0.9, -1.8])

def leg_ik_consts(xml):
    m = mujoco.MjModel.from_xml_path(xml); d = mujoco.MjData(m)
    f0s, jinvs = [], []
    for k, leg in enumerate(LEGS):
        jb = 7 + 3 * k
        gid = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, leg)
        hip = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, leg + "_hip")
        def foot(q3):
            mujoco.mj_resetDataKeyframe(m, d, 0)
            d.qpos[jb:jb + 3] = q3; mujoco.mj_forward(m, d)
            return (d.geom_xpos[gid] - d.xpos[hip]).copy()
        f0 = foot(HOME3_np); e = 1e-3; J = np.zeros((3, 3))
        for j in range(3):
            dq = np.zeros(3); dq[j] = e
            J[:, j] = (foot(HOME3_np + dq) - foot(HOME3_np - dq)) / (2 * e)
        f0s.append(f0); jinvs.append(np.linalg.inv(J))
    return np.array(f0s, np.float32), np.array(jinvs, np.float32)

F0S_np, JINVS_np = leg_ik_consts(SCENE)
print("f0 每腿(x,y,z 相對 hip):\n", np.round(F0S_np, 3))

## 第 4 步：MJX 環境（論文版）

- 動作 12 維 → 論文 CPG → 12 關節角 → `ctrl`（位置伺服，PD 已對齊 kp=90/kd=3）。
- 觀測 76 維：重力(3)+身體線速度(3)+角速度(3)+關節角(12)+關節速度(12)+指令(3)+上一動作(12)+腳觸地(4)+CPG狀態[rx,rx_d,ry,ry_d 各4 + sinθ,cosθ 各4]=24。
- 腳觸地布林用腳掌世界高度 < 3cm 近似。
- **訓練時抗推**：`step` 每 2 秒對機身注入一次隨機水平速度擾動（≤0.6 m/s），policy 學會恢復。

In [ ]:
import functools, jax
from brax.envs.base import Env, State
from mujoco import mjx

CTRL_DT, SIM_DT = 0.02, 0.004
N_FRAMES = int(round(CTRL_DT / SIM_DT))
HOME12 = jnp.array([0.0, 0.9, -1.8] * 4)
HOME3 = jnp.array([0.0, 0.9, -1.8])
KP_NOM, KD_NOM = 90.0, 3.0
KNEE_IDX = [2, 5, 8, 11]
FOOT_CONTACT_H = 0.03
PUSH_EVERY = 100       # 每 100 控制步(=2s) 注入一次隨機速度擾動(抗推訓練)
PUSH_VEL = 0.6         # 速度 kick 上限 (m/s)，隨機水平方向


def apply_pd(m, kp=KP_NOM, kd=KD_NOM):
    m.actuator_gainprm[:, 0] = kp
    m.actuator_biasprm[:, 0] = 0.0
    m.actuator_biasprm[:, 1] = -kp
    m.actuator_biasprm[:, 2] = -kd
    fr = np.full(m.nu, 23.7); fr[KNEE_IDX] = 45.43
    m.actuator_forcerange[:, 0] = -fr; m.actuator_forcerange[:, 1] = fr
    m.actuator_forcelimited[:] = 1
    return m


def _qinv(q): return jnp.array([q[0], -q[1], -q[2], -q[3]])
def _qrot(q, v):
    u = q[1:4]; t = 2.0 * jnp.cross(u, v); return v + q[0] * t + jnp.cross(u, t)
def w2b(quat, v): return _qrot(_qinv(quat), v)


class Go2PaperCpgEnv(Env):
    def __init__(self, f0s, jinvs):
        m = mujoco.MjModel.from_xml_path(SCENE); m.opt.timestep = SIM_DT
        m = apply_pd(m)
        self._mj = m
        self.sys = mjx.put_model(m)
        self._init_q = jnp.array(m.key_qpos[0])
        self._lo = jnp.array(m.actuator_ctrlrange[:, 0])
        self._hi = jnp.array(m.actuator_ctrlrange[:, 1])
        self._f0s = jnp.array(f0s)
        self._jinvs = jnp.array(jinvs)
        self._foot_gid = jnp.array(
            [mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, lg) for lg in LEGS])

    @property
    def observation_size(self): return 76
    @property
    def action_size(self): return 12
    @property
    def backend(self): return "mjx"

    def _sample_cmd(self, rng):
        k1, k2, k3 = jax.random.split(rng, 3)
        vx = jax.random.uniform(k1, (), minval=0.0, maxval=1.0)
        vy = jax.random.uniform(k2, (), minval=-0.3, maxval=0.3)
        wz = jax.random.uniform(k3, (), minval=-1.0, maxval=1.0)
        return jnp.array([vx, vy, wz])

    def _base(self, data):
        quat = data.qpos[3:7]
        gyro = data.qvel[3:6]
        blin = w2b(quat, data.qvel[0:3])
        grav = w2b(quat, jnp.array([0.0, 0.0, -1.0]))
        return quat, gyro, blin, grav

    def _foot_contact(self, data):
        fz = data.geom_xpos[self._foot_gid, 2]              # (4,)
        return (fz < FOOT_CONTACT_H).astype(jnp.float32)

    def _obs(self, data, info):
        _, gyro, blin, grav = self._base(data)
        c = info["cpg"]
        return jnp.concatenate([
            grav, blin, gyro,
            data.qpos[7:19] - HOME12, data.qvel[6:18],
            info["cmd"], info["last_action"], self._foot_contact(data),
            c["rx"], c["rx_d"], c["ry"], c["ry_d"],
            jnp.sin(c["theta"]), jnp.cos(c["theta"]),
        ])

    def reset(self, rng):
        rng, crng = jax.random.split(rng)
        data = mjx.make_data(self.sys).replace(qpos=self._init_q)
        data = mjx.forward(self.sys, data)
        info = {"rng": rng, "cmd": self._sample_cmd(crng),
                "cpg": cpg_init(), "last_action": jnp.zeros(12),
                "step": jnp.zeros((), jnp.int32)}
        obs = self._obs(data, info)
        metrics = {"reward": jnp.zeros(()), "r_lin": jnp.zeros(()),
                   "r_yaw": jnp.zeros(()), "height": jnp.zeros(())}
        return State(data, obs, jnp.zeros(()), jnp.zeros(()), metrics, info)

    def step(self, state, action):
        mux, muy, omega = action_to_cpg_cmd(action)
        cpg = cpg_step(state.info["cpg"], mux, muy, omega, CTRL_DT)
        q_des = cpg_to_joint_targets(cpg, self._f0s, self._jinvs, HOME3)
        ctrl = jnp.clip(q_des, self._lo, self._hi)

        def one(d, _):
            return mjx.step(self.sys, d.replace(ctrl=ctrl)), None
        data, _ = jax.lax.scan(one, state.pipeline_state, None, N_FRAMES)

        # 訓練時每 PUSH_EVERY 步注入一次隨機水平速度擾動（抗推）
        rng, krng = jax.random.split(state.info["rng"])
        step_i = state.info["step"] + 1
        do_push = jnp.mod(step_i, PUSH_EVERY) == 0
        kick = jax.random.uniform(krng, (2,), minval=-PUSH_VEL, maxval=PUSH_VEL)
        qvel = (data.qvel.at[0].add(jnp.where(do_push, kick[0], 0.0))
                          .at[1].add(jnp.where(do_push, kick[1], 0.0)))
        data = data.replace(qvel=qvel)

        info = {**state.info, "cpg": cpg, "last_action": action,
                "rng": rng, "step": step_i}
        obs = self._obs(data, info)
        _, gyro, blin, grav = self._base(data)
        cmd = info["cmd"]
        r_lin = jnp.exp(-((blin[0] - cmd[0]) ** 2 + (blin[1] - cmd[1]) ** 2) / 0.25)
        r_yaw = jnp.exp(-((gyro[2] - cmd[2]) ** 2) / 0.25)
        upright = grav[0] ** 2 + grav[1] ** 2
        height = data.qpos[2]
        height_pen = (height - 0.30) ** 2
        act_rate = jnp.sum((action - state.info["last_action"]) ** 2)
        reward = (1.5 * r_lin + 0.8 * r_yaw - 1.0 * upright
                  - 0.5 * height_pen - 0.05 * act_rate + 0.05)
        done = jnp.where((height < 0.18) | (grav[2] > -0.4), 1.0, 0.0)
        metrics = {"reward": reward, "r_lin": r_lin, "r_yaw": r_yaw, "height": height}
        return state.replace(pipeline_state=data, obs=obs, reward=reward,
                             done=done, metrics=metrics, info=info)

print("paper env defined")

## 第 4.5 步：Domain Randomization（強化版，對齊論文）

比初版多了三項（對應對比實驗一、三的弱點）：
- **摩擦放寬 [0.3, 1.0]**（原 0.5–1.0）→ 更耐滑地面。
- **連桿質量 ±20%**（原 ±10%）、**軀幹隨機加 0~8 kg 負重** → 學會扛重物（論文訓練到 +5kg、泛化到 115% 體重）。
- **外力擾動**：已寫在 env 的 `step`，每 2 秒對機身注入一次隨機水平速度 kick（≤0.6 m/s）→ 學會抗推（論文每 15s、≤0.5 m/s）。

In [ ]:
_mm = mujoco.MjModel.from_xml_path(SCENE)
BASE_ID = mujoco.mj_name2id(_mm, mujoco.mjtObj.mjOBJ_BODY, "base")

def domain_randomize(sys, rng):
    @jax.vmap
    def per_env(rng):
        k1, k2, k3, k4, k5 = jax.random.split(rng, 5)
        geom_friction = sys.geom_friction.at[:, 0].set(
            jax.random.uniform(k1, minval=0.3, maxval=1.0))          # 摩擦放寬 [0.3,1]
        kp = jax.random.uniform(k2, minval=75.0, maxval=105.0)
        kd = jax.random.uniform(k3, minval=2.0, maxval=4.0)
        gain = sys.actuator_gainprm.at[:, 0].set(kp)
        bias = sys.actuator_biasprm.at[:, 1].set(-kp).at[:, 2].set(-kd)
        body_mass = sys.body_mass * jax.random.uniform(
            k4, (sys.nbody,), minval=0.8, maxval=1.2)                # 連桿質量 ±20%
        payload = jax.random.uniform(k5, minval=0.0, maxval=8.0)     # 軀幹加 0~8kg 負重
        body_mass = body_mass.at[BASE_ID].add(payload)
        return geom_friction, gain, bias, body_mass
    gf, gain, bias, bm = per_env(rng)
    in_axes = jax.tree_util.tree_map(lambda x: None, sys)
    in_axes = in_axes.replace(geom_friction=0, actuator_gainprm=0,
                              actuator_biasprm=0, body_mass=0)
    sys = sys.replace(geom_friction=gf, actuator_gainprm=gain,
                      actuator_biasprm=bias, body_mass=bm)
    return sys, in_axes
print("domain_randomize ready")

## Smoke test（開訓練前必跑）

In [ ]:
env = Go2PaperCpgEnv(F0S_np, JINVS_np)
s = jax.jit(env.reset)(jax.random.PRNGKey(0))
print("obs shape:", s.obs.shape, "(應為 (76,))")
s = jax.jit(env.step)(s, jnp.zeros(12))
print("reward:", float(s.reward), "done:", float(s.done), "height:", float(s.metrics["height"]))
print("PASSED" if s.obs.shape == (76,) else "CHECK OBS SIZE")

## 第 5 步：Brax PPO 訓練

動作 12 維、觀測 76 維，比 5 維版稍難，`num_timesteps` 給大一點（8e7）。OOM 就降 `num_envs`。

In [ ]:
import functools, time
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

env = Go2PaperCpgEnv(F0S_np, JINVS_np)
network_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=(256, 256, 128),
    value_hidden_layer_sizes=(256, 256, 256))

train_fn = functools.partial(
    ppo.train, num_timesteps=120_000_000, num_evals=20, episode_length=1000,
    num_envs=2048, batch_size=256, num_minibatches=32, unroll_length=20,
    num_updates_per_batch=4, learning_rate=3e-4, entropy_cost=1e-2,
    discounting=0.97, normalize_observations=True,
    network_factory=network_factory, randomization_fn=domain_randomize, seed=0)

_t0 = time.time(); rewards = []
def progress(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0)); rewards.append((step, r))
    print(f"step {step:>10,}  reward {r:8.2f}  ({time.time()-_t0:.0f}s)")

make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)
print("training done")

In [ ]:
import matplotlib.pyplot as plt
plt.plot([s for s,_ in rewards], [r for _,r in rewards], marker="o")
plt.xlabel("env steps"); plt.ylabel("eval reward"); plt.grid(True); plt.show()

## 看成果：CPU rollout 影片（同時是本機推論樣板）

obs 組法必須和訓練環境逐項一致。

In [ ]:
import mediapy as media

infer = jax.jit(make_inference_fn(params, deterministic=True))

def cpg_init_np():
    return {"rx": np.full(4, 1.5), "rx_d": np.zeros(4),
            "ry": np.full(4, 1.5), "ry_d": np.zeros(4),
            "theta": np.array([0.0, np.pi, np.pi, 0.0])}
PHI_np = np.array([0.0, np.pi, np.pi, 0.0])
PHI_np = PHI_np[None, :] - PHI_np[:, None]

def cpg_step_np(c, mux, muy, omega, dt):
    rx, rxd, ry, ryd, th = (c["rx"].copy(), c["rx_d"].copy(),
                            c["ry"].copy(), c["ry_d"].copy(), c["theta"].copy())
    h = dt / N_CPG_SUB
    for _ in range(N_CPG_SUB):
        rxd += (A_CONV * (A_CONV / 4 * (mux - rx) - rxd)) * h; rx += rxd * h
        ryd += (A_CONV * (A_CONV / 4 * (muy - ry) - ryd)) * h; ry += ryd * h
        rbar = 0.5 * (rx + ry)
        diff = th[None, :] - th[:, None] - PHI_np
        th = th + (2 * np.pi * omega + W_COUP * np.sum(rbar[None, :] * np.sin(diff), 1)) * h
    return {"rx": rx, "rx_d": rxd, "ry": ry, "ry_d": ryd, "theta": th % (2 * np.pi)}

def act_to_cmd_np(a):
    a = np.tanh(a).reshape(4, 3)
    mux = (a[:, 0] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    muy = (a[:, 1] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    om = (a[:, 2] + 1) / 2 * (OMEGA_MAX - OMEGA_MIN) + OMEGA_MIN
    return mux, muy, om

def targets_np(c, f0s, jinvs):
    th = c["theta"]
    fx = 2 * (c["rx"] - MU_MIN) / (MU_MAX - MU_MIN) - 1
    fy = 2 * (c["ry"] - MU_MIN) / (MU_MAX - MU_MIN) - 1
    dx = -D_STEP * fx * np.cos(th); dy = D_STEP * fy * np.cos(th)
    dz = np.where(np.sin(th) > 0, G_C * np.sin(th), G_P * np.sin(th))
    off = np.stack([dx, dy, dz], -1)
    q = np.zeros((4, 3))
    for k in range(4): q[k] = HOME3_np + jinvs[k] @ off[k]
    return q.reshape(12)

def qinv(q): return np.array([q[0], -q[1], -q[2], -q[3]])
def qrot(q, v):
    u = q[1:4]; t = 2 * np.cross(u, v); return v + q[0] * t + np.cross(u, t)
def w2b_np(q, v): return qrot(qinv(q), v)

m = mujoco.MjModel.from_xml_path(SCENE); m.opt.timestep = SIM_DT; m = apply_pd(m)
d = mujoco.MjData(m); mujoco.mj_resetDataKeyframe(m, d, 0)
lo = m.actuator_ctrlrange[:, 0]; hi = m.actuator_ctrlrange[:, 1]
foot_gid = [mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, lg) for lg in LEGS]
ren = mujoco.Renderer(m, 480, 640); cam = mujoco.MjvCamera(); mujoco.mjv_defaultFreeCamera(m, cam)

cmd = np.array([0.6, 0.0, 0.0]); c = cpg_init_np(); last_a = np.zeros(12); frames = []
rng = jax.random.PRNGKey(0)
for i in range(500):
    _, _, blin, grav = None, None, w2b_np(d.qpos[3:7], d.qvel[0:3]), w2b_np(d.qpos[3:7], np.array([0, 0, -1.0]))
    contact = (np.array([d.geom_xpos[g][2] for g in foot_gid]) < FOOT_CONTACT_H).astype(np.float32)
    obs = np.concatenate([grav, blin, d.qvel[3:6], d.qpos[7:19] - np.array([0, 0.9, -1.8] * 4),
                          d.qvel[6:18], cmd, last_a, contact,
                          c["rx"], c["rx_d"], c["ry"], c["ry_d"],
                          np.sin(c["theta"]), np.cos(c["theta"])]).astype(np.float32)
    act = np.array(infer(jnp.asarray(obs), rng))
    mux, muy, om = act_to_cmd_np(act); c = cpg_step_np(c, mux, muy, om, CTRL_DT)
    q_des = targets_np(c, F0S_np, JINVS_np); d.ctrl[:] = np.clip(q_des, lo, hi)
    for _ in range(N_FRAMES): mujoco.mj_step(m, d)
    last_a = act
    if i % 2 == 0:
        cam.lookat[:] = d.qpos[:3]; cam.distance = 2.0; cam.elevation = -20
        ren.update_scene(d, cam); frames.append(ren.render())
print("final x:", round(float(d.qpos[0]), 2), "height:", round(float(d.qpos[2]), 2))
media.show_video(frames, fps=25)

## 第 6 步：存權重下載

In [ ]:
from brax.io import model
model.save_params("cpg_rl_paper_params.pkl", params)
try:
    from google.colab import files; files.download("cpg_rl_paper_params.pkl")
except Exception as e:
    print("左側檔案面板右鍵下載 cpg_rl_paper_params.pkl。", e)

## 帶回本機

用 `local_infer_paper.py`（論文版本機推論）載入 `cpg_rl_paper_params.pkl`，網路設定要和本檔一致
（policy 256/256/128、value 256³、normalize、動作12、觀測76）。CPG/IK 映射與本檔逐行對應。